## SQL 분석 및 피처 엔지니어링
> 파이프라인 위치: 01_data_collection → 02_eda → `03_sql` → 04_modeling

---

### 목적
`02_eda.ipynb`에서 확인한 시차 분석 결과를 반영하여 모델링에 사용할 master_table을 생성한다.

- kamis_egg_retail + ecos_ppi JOIN
- `feed_lag1~3`, `elec_lag1~3` LAG 변수 생성
- regime 더미 컬럼 추가
- master_table DB 적재 및 CSV 저장

In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

from src.db.connection import engine

print(pd.read_sql("SELECT * FROM kamis_egg_retail LIMIT 5", engine))
print(pd.read_sql("SELECT * FROM ecos_ppi LIMIT 5", engine))

  year_month  egg_price
0 2016-01-01       5493
1 2016-02-01       5473
2 2016-03-01       5260
3 2016-04-01       5259
4 2016-05-01       5216
  year_month  egg_ppi  electricity_ppi  feed_ppi  egg_cpi
0 2016-01-01    92.17            98.81     95.69   92.594
1 2016-02-01    85.08            98.81     95.69   91.098
2 2016-03-01    83.52            98.81     94.82   87.269
3 2016-04-01    89.81            98.81     94.82   86.773
4 2016-05-01    85.53            98.81     94.82   84.645


- Python에서 `shift()`로 LAG를 생성할 수도 있지만 데이터 정합성과 재현성을 높이기 위해     
SQL의 `LAG() OVER (ORDER BY year_month)`를 사용했다.    
DB 단계에서 피처를 확정하면 이후 모델링 코드가 단순해지고 동일한 피처를 항상 동일한 방식으로 재현할 수 있다.     

In [2]:
query_join = """
SELECT
    a.year_month,
    a.egg_price,
    b.feed_ppi,
    b.electricity_ppi,
    b.egg_ppi,
    b.egg_cpi
FROM kamis_egg_retail a
JOIN ecos_ppi b ON a.year_month = b.year_month
ORDER BY a.year_month
"""

df_join = pd.read_sql(query_join, engine)

print("JOIN 결과 행 수:", df_join.shape)  
print(df_join.head())
print(df_join.isnull().sum())

JOIN 결과 행 수: (120, 6)
  year_month  egg_price  feed_ppi  electricity_ppi  egg_ppi  egg_cpi
0 2016-01-01       5493     95.69            98.81    92.17   92.594
1 2016-02-01       5473     95.69            98.81    85.08   91.098
2 2016-03-01       5260     94.82            98.81    83.52   87.269
3 2016-04-01       5259     94.82            98.81    89.81   86.773
4 2016-05-01       5216     94.82            98.81    85.53   84.645
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
dtype: int64


- 02_eda 시계열 차트에서 2021년을 기점으로 달걀 가격의 평상시 수준이    
구조적으로 상향된 것이 육안으로 확인됐다.    
동일한 원가 수준에서도 시기에 따라 가격 기준이 다를 수 있으므로     
이를 모델에 반영하기 위해 2021년 이후를 regime=1로 구분하는 더미변수를 생성했다.    
수치 검증은 04_modeling에서 수행한다.

In [3]:
query_lag = """
SELECT
    year_month,
    egg_price,
    feed_ppi,
    electricity_ppi,
    egg_ppi,
    egg_cpi,
    LAG(feed_ppi, 1) OVER (ORDER BY year_month) AS feed_lag1,
    LAG(feed_ppi, 2) OVER (ORDER BY year_month) AS feed_lag2,
    LAG(feed_ppi, 3) OVER (ORDER BY year_month) AS feed_lag3,
    LAG(electricity_ppi, 1) OVER (ORDER BY year_month) AS elec_lag1,
    LAG(electricity_ppi, 2) OVER (ORDER BY year_month) AS elec_lag2,
    LAG(electricity_ppi, 3) OVER (ORDER BY year_month) AS elec_lag3,
    CASE WHEN EXTRACT(YEAR FROM year_month::date) >= 2021 THEN 1 ELSE 0 END AS regime
FROM (
    SELECT
        a.year_month,
        a.egg_price,
        b.feed_ppi,
        b.electricity_ppi,
        b.egg_ppi,
        b.egg_cpi
    FROM kamis_egg_retail a
    JOIN ecos_ppi b ON a.year_month = b.year_month
    ORDER BY a.year_month
)
"""

df_master = pd.read_sql(query_lag, engine)
print("LAG 변수 추가 후:", df_master.shape) 
print(df_master.head())
print(df_master.isnull().sum())

LAG 변수 추가 후: (120, 13)
  year_month  egg_price  feed_ppi  electricity_ppi  egg_ppi  egg_cpi  \
0 2016-01-01       5493     95.69            98.81    92.17   92.594   
1 2016-02-01       5473     95.69            98.81    85.08   91.098   
2 2016-03-01       5260     94.82            98.81    83.52   87.269   
3 2016-04-01       5259     94.82            98.81    89.81   86.773   
4 2016-05-01       5216     94.82            98.81    85.53   84.645   

   feed_lag1  feed_lag2  feed_lag3  elec_lag1  elec_lag2  elec_lag3  regime  
0        NaN        NaN        NaN        NaN        NaN        NaN       0  
1      95.69        NaN        NaN      98.81        NaN        NaN       0  
2      95.69      95.69        NaN      98.81      98.81        NaN       0  
3      94.82      95.69      95.69      98.81      98.81      98.81       0  
4      94.82      94.82      95.69      98.81      98.81      98.81       0  
year_month         0
egg_price          0
feed_ppi           0
electricity_p

> LAG 변수 생성으로 인해 초기 3행에 결측치가 발생하며 이를 다음 단계에서 제거한다.

In [4]:
df_master = df_master.dropna().reset_index(drop=True)

print("결측치 제거 후:", len(df_master), "행")  
print(df_master.isnull().sum())  

결측치 제거 후: 117 행
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
feed_lag1          0
feed_lag2          0
feed_lag3          0
elec_lag1          0
elec_lag2          0
elec_lag3          0
regime             0
dtype: int64


In [5]:
df_master.to_sql("master_table", engine, if_exists="replace", index=False)

# 검증 
df_master_db = pd.read_sql("SELECT * FROM master_table", engine)

print("적재 전 행 수:", len(df_master))
print("적재 후 행 수:", len(df_master_db))
print("일치 여부:", len(df_master) == len(df_master_db))
print(df_master_db.isnull().sum())

적재 전 행 수: 117
적재 후 행 수: 117
일치 여부: True
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
feed_lag1          0
feed_lag2          0
feed_lag3          0
elec_lag1          0
elec_lag2          0
elec_lag3          0
regime             0
dtype: int64


> 적재 전후 행 수 일치 및 결측치 0건을 확인했다.     
> 이후 04_modeling에서 master_table을 직접 조회하여 분석을 수행한다.

In [6]:
df_master.to_csv("../data/processed/master_table.csv", index=False)
print("마스터 테이블 CSV 저장 완료")

마스터 테이블 CSV 저장 완료
